# Features extraction

In [1]:
# imports

from pathlib import Path # handling file paths

import numpy as np # numerical operations
import pandas as pd # CSV files

from scipy.signal import welch # spectral density estimation
from scipy.integrate import trapezoid # integration

import matplotlib.pyplot as plt # plotting

from tqdm.auto import tqdm # progress bars

from PIL import Image # display images in notebook

## 1. Settings

In [2]:
# Paths
DATA_PATH = Path("data") / "adhdata.csv"
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = Path("outputs") # directory for outputs
OUTPUT_DIR.mkdir(parents=True, exist_ok=True) # create output directory if it doesn't exist
FIGURES_DIR = OUTPUT_DIR / "figures"
TABS_DIR = OUTPUT_DIR / "tabs"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABS_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset exists:", DATA_PATH.exists())

Dataset exists: True


In [3]:
def save_figure(filename, dpi=300):
    """
    Save the current matplotlib figure inside outputs/figures.
    """
    output_path = FIGURES_DIR / filename
    plt.savefig(output_path, dpi=dpi, bbox_inches="tight")
    print("Figure saved to:", output_path)

In [4]:
# Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (2166383, 21)


,Fp1,Fp2,F3,F4,C3,C4,P3,P4,O1,O2,...,F8,T7,T8,P7,P8,Fz,Cz,Pz,Class,ID
0,261.0,402.0,16.0,261.0,126.0,384.0,126.0,236.0,52.0,236.0,...,16.0,200.0,494.0,126.0,236.0,121.0,367.0,121.0,ADHD,v10p
1,121.0,191.0,-94.0,85.0,16.0,200.0,126.0,52.0,347.0,273.0,...,-57.0,126.0,347.0,52.0,52.0,15.0,121.0,-19.0,ADHD,v10p
2,-55.0,85.0,-204.0,15.0,-57.0,200.0,52.0,126.0,236.0,200.0,...,-94.0,126.0,420.0,52.0,126.0,-55.0,261.0,85.0,ADHD,v10p
3,191.0,85.0,52.0,50.0,89.0,236.0,163.0,89.0,89.0,89.0,...,-57.0,236.0,420.0,126.0,126.0,15.0,85.0,-55.0,ADHD,v10p
4,-55.0,-125.0,-204.0,-160.0,-204.0,16.0,-241.0,-241.0,89.0,16.0,...,-131.0,89.0,310.0,-57.0,52.0,-55.0,15.0,-336.0,ADHD,v10p


In [5]:
# EEG channels, frequency bands and regions

EEG_CHANNELS = [
    "Fp1", "Fp2", "F3", "F4",
    "C3", "C4",
    "P3", "P4",
    "O1", "O2",
    "F7", "F8",
    "T7", "T8",
    "P7", "P8",
    "Fz", "Cz", "Pz"
]

FS = 128  # sampling frequency in Hz

FREQ_BANDS = {
    "delta": (1, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 30)
}

REGIONS = {
    "frontal": ["Fp1", "Fp2", "F3", "F4", "F7", "F8", "Fz"],
    "central": ["C3", "C4", "Cz"],
    "parietal": ["P3", "P4", "P7", "P8", "Pz"],
    "temporal": ["T7", "T8"],
    "occipital": ["O1", "O2"]
}

## 2. Extracting Features

### Power Spectral Density estimation using Welch's method

EEG signals are time series: for each subject and each electrode, we have a sequence of voltage values changing over time. EEG is often analyzed in the frequency domain.

The goal of spectral analysis is to decompose a signal into sinusoidal components at different frequencies. Instead of asking only *how the signal changes over time*, we ask *how much power does the signal contain at each frequency*.

This is particularly useful for EEG because brain activity is often described in terms of frequency bands, such as delta, theta, alpha and beta.

### From time domain to frequency domain

Let $x[n]$ be a discrete EEG signal, where $n$ is the sample index. The signal is sampled at a fixed sampling frequency $f_s$, which in this dataset is: $f_s = 128 \text{ Hz}$

This means that 128 samples are recorded every second.

The Fourier transform allows us to represent the signal as a combination of sinusoidal waves with different frequencies. For a discrete signal, this is done using the Discrete Fourier Transform (DFT): $X[k] = \sum_{n=0}^{N-1} x[n] e^{-j 2\pi kn/N}$
where:
- $x[n]$ is the signal in the time domain;
- $X[k]$ is the frequency-domain representation;
- $N$ is the number of samples;
- $k$ indexes the frequency bins.

The squared magnitude of the Fourier coefficients gives information about the power of the signal at each frequency.
However, applying a single Fourier transform to the whole EEG signal can produce a noisy estimate, because EEG is non-stationary and contains random fluctuations.

### Power Spectral Density

The Power Spectral Density (PSD) describes how the power of a signal is distributed across frequencies. The unit of a PSD is typically $\frac{U^2}{Hz}$
where $U$ is the unit of the original signal, for example microvolts. This means that the PSD represents power per unit of frequency.

### Welch's method

In this project, the PSD is estimated using Welch's method. Welch's method is a standard approach for obtaining a more stable estimate of the PSD.
The main idea is:
1. split the signal into shorter segments;
2. apply a window function to each segment;
3. compute the Fourier transform of each windowed segment;
4. compute the periodogram of each segment;
5. average the periodograms across segments.

This reduces random fluctuations in the PSD estimate.
If the signal is divided into $K$ segments, Welch's PSD estimate can be written as:
$
\hat{S}_{xx}(f) = \frac{1}{K} \sum_{k=1}^{K} P_k(f)
$
where:
- $\hat{S}_{xx}(f)$ is the estimated PSD;
- $P_k(f)$ is the periodogram of segment $k$;
- $K$ is the number of segments.

A periodogram is approximately the squared magnitude of the Fourier transform of a segment:
$
P_k(f) \propto |X_k(f)|^2
$
where $X_k(f)$ is the Fourier transform of the $k$-th segment.

Averaging multiple periodograms produces a smoother and more reliable PSD estimate than using only one Fourier transform of the entire signal.

### Window length and overlap

In this notebook, Welch's method is applied with $n_{\text{perseg}} = 512$
Since the sampling frequency is $f_s = 128 \text{ Hz}$ each Welch window has duration of  $T = \frac{512}{128} = 4 \text{ seconds}$.

The frequency resolution is $\Delta f = \frac{f_s}{n_{\text{perseg}}}$, therefore
$\Delta f = \frac{128}{512} = 0.25 \text{ Hz}$.

A 50% overlap is used $n_{\text{overlap}} = 256$.

This means that consecutive windows share half of their samples. Overlap increases the number of segments used for averaging, improving the stability of the PSD estimate.

### EEG frequency bands

After estimating the PSD, band powers are computed by integrating the PSD within standard EEG frequency ranges.

The frequency bands used here are:

- $\delta: 1-4 \text{ Hz}$
- $\theta: 4-8 \text{ Hz}$
- $\alpha: 8-12 \text{ Hz}$
- $\beta: 12-30 \text{ Hz}$

For a given band $[f_1, f_2]$, the band power is computed as the area under the PSD curve in that frequency interval:
$P_{\text{band}} = \int_{f_1}^{f_2} PSD(f)\,df$

In practice, since the PSD is computed at discrete frequency bins, the integral is approximated numerically:

$P_{\text{band}} \approx \sum_{f=f_1}^{f_2} PSD(f)\Delta f$ or, more accurately, using trapezoidal integration.

For example, theta power is computed as $P_{\theta} = \int_{4}^{8} PSD(f)\,df$
and beta power as $P_{\beta} = \int_{12}^{30} PSD(f)\,df$


In [6]:
# Welch parameters

NPERSEG = 512      # 4 seconds at 128 Hz
NOVERLAP = 256     # 50% overlap

print("Window duration:", NPERSEG / FS, "seconds")
print("Frequency resolution:", FS / NPERSEG, "Hz")

Window duration: 4.0 seconds
Frequency resolution: 0.25 Hz


In [7]:
# functions for feature extraction

def compute_band_powers(signal, fs=128, nperseg=512, noverlap=256):
    """
    Compute absolute band powers from a 1D EEG signal using Welch PSD.

    Returns one value for each frequency band.
    """
    # compute PSD using Welch's method, returns frequencies and power spectral density values
    freqs, psd = welch(
        signal,
        fs=fs,
        nperseg=nperseg,
        noverlap=noverlap
    ) 
    
    band_powers = {} #empty dictionary

    # loop through each frequency band
    for band_name, (fmin, fmax) in FREQ_BANDS.items(): 

        # select frequencies of the current band
        band_mask = (freqs >= fmin) & (freqs < fmax) # boolean mask

        # area under the PSD curve in this band (band power)
        # trapezoid method
        power = trapezoid(
            psd[band_mask],
            freqs[band_mask]
        )

        band_powers[band_name] = power # store result

    return band_powers



def extract_subject_features(subject_df, subject_id):
    """
    Extract EEG spectral features for a single subject.

    Output:
    one dictionary = one row in the final feature table.
    """
    features = {}

    subject_class = subject_df["Class"].iloc[0] # get class label

    features["ID"] = subject_id
    features["Class"] = subject_class
    features["n_samples"] = len(subject_df)
    features["duration_sec"] = len(subject_df) / FS # duration in seconds

    channel_band_powers = {}

    # channel-level features

    # loop through each EEG channel and compute band powers
    for channel in EEG_CHANNELS: 
        signal = subject_df[channel].values # 1D array of EEG values for this channel

        band_powers = compute_band_powers(
            signal,
            fs=FS,
            nperseg=NPERSEG,
            noverlap=NOVERLAP
        )

        channel_band_powers[channel] = band_powers

        # store band powers as features
        for band_name, power in band_powers.items():
            features[f"{channel}_{band_name}_power"] = power

        # compute theta/beta ratio for this channel
        theta = band_powers["theta"]
        beta = band_powers["beta"]

        # NaN if beta is zero
        features[f"{channel}_theta_beta_ratio"] = theta / beta if beta != 0 else np.nan

    # regional features
    for region_name, region_channels in REGIONS.items():
        for band_name in FREQ_BANDS.keys():
            values = [
                channel_band_powers[ch][band_name]
                for ch in region_channels
            ]

            # average across channel in this region
            features[f"{region_name}_{band_name}_power_mean"] = np.mean(values)

        region_theta = features[f"{region_name}_theta_power_mean"]
        region_beta = features[f"{region_name}_beta_power_mean"]

        features[f"{region_name}_theta_beta_ratio"] = (
            region_theta / region_beta if region_beta != 0 else np.nan
        )

    return features

In [8]:
## test on one subject and all channels

# test_id = df["ID"].iloc[0]
# test_channel = "Fz"

# test_signal = df.loc[df["ID"] == test_id, test_channel].values

# print("Test subject:", test_id)
# print("Signal length:", len(test_signal))
# print("Duration:", len(test_signal) / FS, "seconds")

# test_band_powers = compute_band_powers(
#     test_signal,
#     fs=FS,
#     nperseg=NPERSEG,
#     noverlap=NOVERLAP
# )

# test_band_powers

# test_subject_df = df[df["ID"] == test_id]

# test_features = extract_subject_features(
#     test_subject_df,
#     subject_id=test_id
# )

# print("Number of features:", len(test_features))
# test_features


# test_features = extract_subject_features(
#     test_subject_df,
#     subject_id=test_id
# )

# print("Number of features:", len(test_features))

# for key in list(test_features.keys())[:30]:
#     print(key, ":", test_features[key])

### Theta/Beta Ratio

In ADHD EEG research, the theta/beta ratio, $ TBR = \frac{P_{\theta}}{P_{\beta}}$, is often considered an important spectral feature.
In this project, theta/beta ratio is computed both for individual electrodes and for broader scalp regions, such as frontal, temporal, parietal and occipital regions.

### Feature extraction logic
The complete feature extraction pipeline is therefore:

$
\text{Raw EEG signal}
\rightarrow
\text{Welch PSD}
\rightarrow
\text{Band powers}
\rightarrow
\text{Theta/Beta ratio}
\rightarrow
\text{Subject-level feature table}
$

For each subject, the final output is one row containing spectral EEG features extracted from the full recording. 

In [9]:
all_features = []

subject_ids = df["ID"].unique()

for subject_id in tqdm(subject_ids):
    subject_df = df[df["ID"] == subject_id]

    subject_features = extract_subject_features(
        subject_df,
        subject_id=subject_id
    )

    all_features.append(subject_features)

features_df = pd.DataFrame(all_features)

print("Feature table shape:", features_df.shape)
display(features_df.head())

  0%|          | 0/121 [00:00<?, ?it/s]

Feature table shape: (121, 124)


,ID,Class,n_samples,duration_sec,Fp1_delta_power,Fp1_theta_power,Fp1_alpha_power,Fp1_beta_power,Fp1_theta_beta_ratio,Fp2_delta_power,...,temporal_delta_power_mean,temporal_theta_power_mean,temporal_alpha_power_mean,temporal_beta_power_mean,temporal_theta_beta_ratio,occipital_delta_power_mean,occipital_theta_power_mean,occipital_alpha_power_mean,occipital_beta_power_mean,occipital_theta_beta_ratio
0,v10p,ADHD,14304,111.750000,20764.347948,4433.165153,1163.116779,1220.075523,3.633517,19985.184042,...,5839.205234,1832.689764,2206.839430,2033.163635,0.901398,5313.211114,2515.280960,1239.193542,2047.478172,1.228478
1,v12p,ADHD,17604,137.531250,35697.951547,7797.099598,1427.690126,1028.902332,7.578076,29986.175718,...,23000.755768,3191.807975,1139.959981,908.476562,3.513363,9076.262423,3731.706544,1414.186873,2011.651626,1.855046
2,v14p,ADHD,17562,137.203125,124322.835760,34554.128417,4828.326846,5765.890863,5.992852,198052.819591,...,32551.843934,5516.419259,2188.738526,2493.216978,2.212571,21449.856904,6070.162746,2775.817586,3659.736876,1.658634
3,v15p,ADHD,43252,337.906250,25609.106999,7425.362020,2163.548768,1443.360033,5.144497,21041.122924,...,13208.390881,3255.140409,2278.065705,1660.059682,1.960857,14728.933087,4269.300202,1999.343488,1661.070205,2.570211
4,v173,ADHD,24241,189.382812,6962.611895,3144.051753,1982.601543,6955.012349,0.452056,7648.534786,...,2837.889255,1387.619299,730.351150,2373.689547,0.584583,5284.728897,2475.850442,1332.758218,4059.060983,0.609956


In [10]:
# check

print(features_df["Class"].value_counts())
print("Missing values:", features_df.isna().sum().sum())
display(features_df.describe().T.head(20))

Class
ADHD       61
Control    60
Name: count, dtype: int64
Missing values: 0


,count,mean,std,min,25%,50%,75%,max
n_samples,121.0,17903.991736,6207.400652,7983.000000,13697.000000,16697.000000,20115.000000,43252.000000
duration_sec,121.0,139.874935,48.495318,62.367188,107.007812,130.445312,157.148438,337.906250
Fp1_delta_power,121.0,34026.344040,36725.488086,4194.847201,11899.265040,22802.072562,36503.291494,230267.147330
Fp1_theta_power,121.0,6889.097820,5194.779530,1205.053716,3511.074492,5548.236635,7797.099598,34554.128417
Fp1_alpha_power,121.0,1856.172810,1155.791063,359.744157,1177.846126,1626.570852,2151.159206,10027.868624
Fp1_beta_power,121.0,2680.845596,1958.913943,387.611111,1533.871210,2237.456171,2942.909975,14498.602768
Fp1_theta_beta_ratio,121.0,2.987080,1.852366,0.452056,1.673695,2.436126,3.768611,9.991347
Fp2_delta_power,121.0,35701.836683,42774.867382,4470.606948,11513.951109,21592.732328,36365.465802,274859.956372
Fp2_theta_power,121.0,6935.394323,6256.187580,1228.287667,3282.747117,5170.842576,8312.179089,48115.131534
Fp2_alpha_power,121.0,1907.291760,1500.358719,474.114861,1108.733697,1571.832059,2076.821936,12993.787032


## 3. Saving new dataset

In [11]:
OUTPUT_PATH = OUTPUT_DIR / "eeg_features_data.csv"

features_df.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)

Saved to: outputs\eeg_features_data.csv
